
# FIQA / BEIR RAG Retriever Comparison

This notebook compares multiple **RAG-style retriever stacks** on the **BEIR FIQA** dataset using classic IR metrics:

- `avg_precision@5`, `avg_recall@5`, `avg_ndcg@5`
- `avg_precision@10`, `avg_recall@10`, `avg_ndcg@10`

We evaluate:

1. **BEIR-native retrievers**
   - BM25 (lexical)
   - Dense SBERT
   - Hybrid (lexical + dense)
2. **Haystack** BM25 retriever
3. **LangChain** FAISS dense retriever
4. **LlamaIndex** VectorStoreIndex retriever

All are normalized into a **BEIR-style results format** and evaluated via `EvaluateRetrieval` so that metrics are directly comparable.



## 0. Install dependencies

Uncomment and run this cell if you don't already have the libraries installed.


In [1]:
# srun -A p30309 -p gengpu --gres=gpu:2 --time=03:59:00 --mem=400G --pty /bin/bash
import socket
print(socket.gethostname())

qgpu3007


In [ ]:
# Optional: Set HuggingFace API key (only needed for private/gated models)
# For public models like 'sentence-transformers/msmarco-distilbert-base-tas-b', 
# no API key is required. However, if you want to use private models or 
# increase rate limits, set your API key here.

import os

# Method 1: Set via environment variable (recommended)
# You can also set this before starting Jupyter:
# export HUGGINGFACE_HUB_TOKEN="your_token_here"
# or
# export HF_TOKEN="your_token_here"

# Method 2: Set programmatically in this notebook (uncomment and add your token)
HF_TOKEN = "<your_huggingface_token_here>" # "your_huggingface_token_here"
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_TOKEN"] = HF_TOKEN  # Alternative name

# Method 3: Login using huggingface_hub (alternative)
# from huggingface_hub import login
# login(token="your_token_here")

# Verify if token is set (optional check)
if os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN"):
    print("✅ HuggingFace API token is set")
else:
    print("ℹ️  HuggingFace API token not set (not required for public models)")


✅ HuggingFace API token is set


In [3]:
# Download NLTK tokenizer data (required for rank-bm25 fallback)
import nltk
try:
    nltk.data.find('tokenizers/punkt_tab')
    print("✅ NLTK punkt_tab tokenizer already available")
except LookupError:
    print("📥 Downloading NLTK punkt_tab tokenizer...")
    nltk.download('punkt_tab', quiet=False)
    print("✅ NLTK punkt_tab tokenizer downloaded")


✅ NLTK punkt_tab tokenizer already available


In [ ]:

# !pip install beir sentence-transformers pyserini
# !pip install farm-haystack
# !pip install "llama-index>=0.10.0"
# !pip install "langchain>=0.2.0" "langchain-community" datasets faiss-cpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.8/178.8 MB 31.6 MB/s  0:00:050m eta 0:00:010:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of pyserini to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.5/178.5 MB 36.8 MB/s  0:00:040m eta 0:00:010:01:02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.5/178.5 MB 32.6 MB/s  0:00:050m eta 0:00:010:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.6/194.6 MB 26.9 MB/s  0:00:070m eta 0:00:010:01:01
  Installing build dependencies ... done
  Get


## 1. Common BEIR setup and metric helpers


In [4]:

import os
from typing import Dict, List
from pathlib import Path

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

# -----------------------------
# 1. Load FIQA dataset
# -----------------------------
def load_fiqa(split: str = "test"):
    # Use the full URL for FIQA dataset
    url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip"
    data_dir = Path("./datasets")
    data_dir.mkdir(exist_ok=True)
    
    # Download and unzip
    data_path = util.download_and_unzip(url, str(data_dir))
    
    # Find the actual data directory (might be nested)
    data_path = Path(data_path)
    if (data_path / "fiqa").exists():
        data_path = data_path / "fiqa"
    elif not (data_path / "corpus.jsonl").exists():
        # Search for corpus.jsonl
        corpus_files = list(data_path.rglob("corpus.jsonl"))
        if corpus_files:
            data_path = corpus_files[0].parent
    
    # Load the dataset
    corpus, queries, qrels = GenericDataLoader(str(data_path)).load(split=split)
    return corpus, queries, qrels

K_VALUES = [5, 10]


# -----------------------------
# 2. Metric helpers
# -----------------------------
def extract_metrics_dict(name, ndcg, map_, recall, prec, k_values=K_VALUES):
    """
    Convert BEIR metrics dicts into a flat dict with keys like:
        avg_precision@5, avg_recall@5, avg_ndcg@5, ...
    Handles different BEIR API versions with different key formats.
    """
    out = {"pipeline": name}
    
    # Helper function to safely get metric value trying multiple key formats
    def get_metric(metric_dict, k, key_variants):
        """Try multiple key formats, return 0.0 if none found."""
        for key in key_variants:
            val = metric_dict.get(key)
            if val is not None:  # Check for None, not falsy (0.0 is valid!)
                print(f"Found {key} with value {val}")
                return val
        return 0.0
    
    for k in k_values:
        # Precision - try different key formats
        prec_val = get_metric(prec, k, [f"P@{k}", f"P-{k}", f"precision@{k}", f"Precision@{k}"])
        out[f"avg_precision@{k}"] = prec_val
        
        # Recall - try different key formats
        recall_val = get_metric(recall, k, [f"R@{k}", f"R-{k}", f"recall@{k}", f"Recall@{k}"])
        out[f"avg_recall@{k}"] = recall_val
        
        # NDCG - try different key formats
        ndcg_val = get_metric(ndcg, k, [f"NDCG@{k}", f"NDCG-{k}", f"ndcg@{k}", f"Ndcg@{k}"])
        out[f"avg_ndcg@{k}"] = ndcg_val
        
        # MAP - try different key formats
        map_val = get_metric(map_, k, [f"MAP@{k}", f"MAP-{k}", f"map@{k}", f"Map@{k}"])
        out[f"avg_map@{k}"] = map_val
    return out


def print_metrics_table(all_results: List[Dict], k_values=K_VALUES):
    """
    Pretty-print a comparison table across pipelines with aligned columns.
    """
    if not all_results:
        print("No results to display.")
        return

    # Build header
    header = ["pipeline"]
    for k in k_values:
        header += [f"P@{k}", f"R@{k}", f"nDCG@{k}"]
    
    # Calculate column widths
    col_widths = []
    for i, col in enumerate(header):
        max_width = len(col)
        # Check all rows for this column
        for res in all_results:
            if i == 0:  # pipeline column
                max_width = max(max_width, len(res["pipeline"]))
            else:
                # For metric columns, format is always 6 chars (0.0000)
                max_width = max(max_width, 6)
        col_widths.append(max_width + 2)  # Add padding
    
    # Print header
    header_str = "".join([f"{header[i]:<{col_widths[i]}}" for i in range(len(header))])
    print(header_str)
    print("-" * len(header_str))
    
    # Print rows
    for res in all_results:
        row = [res["pipeline"]]
        for k in k_values:
            row += [
                f"{res.get(f'avg_precision@{k}', 0.0):.4f}",
                f"{res.get(f'avg_recall@{k}', 0.0):.4f}",
                f"{res.get(f'avg_ndcg@{k}', 0.0):.4f}",
            ]
        row_str = "".join([f"{row[i]:<{col_widths[i]}}" for i in range(len(row))])
        print(row_str)


/home/kzy816/.conda/envs/rag_app/lib/python3.11/site-packages/beir/util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm



## 2. BEIR-native pipelines (BM25 / dense / hybrid)


In [5]:

from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from beir.retrieval.models import SentenceBERT

def eval_beir_pipelines(corpus, queries, qrels, k_values=K_VALUES):
    metrics = []
    bm25_results = None
    dense_results = None

    # 1) BM25 lexical - Try Elasticsearch first, fallback to rank-bm25
    # Note: BEIR's ElasticSearch is incompatible with Elasticsearch 9.x
    # It will automatically fall back to rank-bm25 (pure Python, no Elasticsearch needed)
    try:
        from beir.retrieval.search.lexical.elastic_search import ElasticSearch
        import elasticsearch
        
        # Check if Elasticsearch version is compatible (BEIR works with 7.x/8.x, not 9.x)
        es_version = elasticsearch.__version__.split('.')[0]
        if int(es_version) >= 9:
            raise ImportError(f"Elasticsearch {elasticsearch.__version__} is incompatible with BEIR. Using rank-bm25 fallback instead.")
        
        # ElasticSearch requires es_credentials as a dictionary
        es_credentials = {
            "index_name": "fiqa-bm25",
            "hostname": "localhost:9200",  # Include port in hostname
            "language": "english",
            "keys": {
                "title": "title",
                "body": "text"
            },
            "number_of_shards": "default",
            "timeout": 100,
            "retry_on_timeout": True,
            "maxsize": 25
        }
        
        bm25_search = ElasticSearch(es_credentials=es_credentials)
        bm25_retriever = EvaluateRetrieval(bm25_search)
        
        bm25_results = bm25_retriever.retrieve(corpus, queries)
        bm25_ndcg, bm25_map, bm25_recall, bm25_prec = bm25_retriever.evaluate(
            qrels, bm25_results, k_values
        )
        metrics.append(
            extract_metrics_dict("beir_bm25_elasticsearch", bm25_ndcg, bm25_map, bm25_recall, bm25_prec, k_values)
        )
        print("✅ BM25 using Elasticsearch")
    except Exception as e1:
        # Elasticsearch failed (version incompatibility or not running) - use rank-bm25 fallback
        if "timeout" in str(e1) or "maxsize" in str(e1) or "incompatible" in str(e1):
            print(f"⚠️ Elasticsearch incompatible with this version. Using rank-bm25 fallback (no Elasticsearch needed).")
        else:
            print(f"⚠️ Elasticsearch BM25 failed: {e1}. Using rank-bm25 fallback.")
        # Fallback: Use rank-bm25 (pure Python, no Elasticsearch needed)
        try:
            from rank_bm25 import BM25Okapi
            import nltk
            from nltk.tokenize import word_tokenize
            
            # Download NLTK data if needed (should already be downloaded, but check as fallback)
            try:
                nltk.data.find('tokenizers/punkt_tab')
            except LookupError:
                print("⚠️ punkt_tab not found, downloading...")
                nltk.download('punkt_tab', quiet=False)
            
            # Build BM25 index
            tokenized_corpus = []
            doc_ids = []
            for doc_id, doc in corpus.items():
                text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
                tokens = word_tokenize(text.lower())
                tokenized_corpus.append(tokens)
                doc_ids.append(doc_id)
            
            bm25 = BM25Okapi(tokenized_corpus)
            
            # Retrieve for each query
            results = {}
            for qid, query_text in queries.items():
                query_tokens = word_tokenize(query_text.lower())
                scores = bm25.get_scores(query_tokens)
                # Get top-k documents
                top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:max(k_values)]
                results[qid] = {doc_ids[i]: float(scores[i]) for i in top_indices}
            
            # Evaluate
            dummy_eval = EvaluateRetrieval(None)
            bm25_ndcg, bm25_map, bm25_recall, bm25_prec = dummy_eval.evaluate(
                qrels, results, k_values
            )
            metrics.append(
                extract_metrics_dict("beir_bm25_rankbm25", bm25_ndcg, bm25_map, bm25_recall, bm25_prec, k_values)
            )
            bm25_results = results
            print("✅ BM25 using rank-bm25 (pure Python)")
        except Exception as e2:
            print(f"⚠️ rank-bm25 also failed: {e2}")
            print("   Skipping BM25. Install: pip install rank-bm25 nltk")

    # 2) Dense SBERT
    try:
        sbert_model = SentenceBERT("sentence-transformers/msmarco-distilbert-base-tas-b")
        dense_search = DRES(sbert_model, batch_size=128)
        dense_retriever = EvaluateRetrieval(dense_search, score_function="cos_sim")

        dense_results = dense_retriever.retrieve(corpus, queries)
        dense_ndcg, dense_map, dense_recall, dense_prec = dense_retriever.evaluate(
            qrels, dense_results, k_values
        )
        metrics.append(
            extract_metrics_dict("beir_dense_sbert", dense_ndcg, dense_map, dense_recall, dense_prec, k_values)
        )
        print("✅ Dense SBERT pipeline completed")
    except Exception as e:
        print(f"⚠️ Dense SBERT pipeline failed: {e}")

    # 3) Hybrid (lexical + dense) - only if both succeeded
    if bm25_results is not None and dense_results is not None:
        try:
            # Manually combine BM25 and dense results
            # alpha controls the weight: 0.5 = equal weight, 0.0 = only dense, 1.0 = only BM25
            alpha = 0.5
            
            def combine_results(bm25_scores, dense_scores, alpha):
                """Combine BM25 and dense scores using weighted sum after normalization."""
                # Normalize scores to [0, 1] range for each query
                def normalize_scores(scores_dict):
                    if not scores_dict:
                        return scores_dict
                    max_score = max(scores_dict.values()) if scores_dict.values() else 1.0
                    min_score = min(scores_dict.values()) if scores_dict.values() else 0.0
                    if max_score == min_score:
                        return {k: 0.5 for k in scores_dict.keys()}
                    return {k: (v - min_score) / (max_score - min_score) 
                            for k, v in scores_dict.items()}
                
                # Get all document IDs
                all_docs = set(bm25_scores.keys()) | set(dense_scores.keys())
                
                # Normalize both score sets
                norm_bm25 = normalize_scores(bm25_scores)
                norm_dense = normalize_scores(dense_scores)
                
                # Combine with weighted sum
                combined = {}
                for doc_id in all_docs:
                    bm25_score = norm_bm25.get(doc_id, 0.0)
                    dense_score = norm_dense.get(doc_id, 0.0)
                    combined[doc_id] = alpha * bm25_score + (1 - alpha) * dense_score
                
                return combined
            
            # Combine results for each query
            hybrid_results = {}
            for qid in queries.keys():
                if qid in bm25_results and qid in dense_results:
                    hybrid_results[qid] = combine_results(
                        bm25_results[qid], 
                        dense_results[qid], 
                        alpha
                    )
                elif qid in bm25_results:
                    hybrid_results[qid] = bm25_results[qid]
                elif qid in dense_results:
                    hybrid_results[qid] = dense_results[qid]
            
            # Evaluate hybrid results
            dummy_eval = EvaluateRetrieval(None)
            hybrid_ndcg, hybrid_map, hybrid_recall, hybrid_prec = dummy_eval.evaluate(
                qrels, hybrid_results, k_values
            )
            metrics.append(
                extract_metrics_dict("beir_hybrid_0.5", hybrid_ndcg, hybrid_map, hybrid_recall, hybrid_prec, k_values)
            )
            print("✅ Hybrid pipeline completed")
        except Exception as e:
            print(f"⚠️ Hybrid pipeline failed: {e}")
    
    return metrics




## 3. Haystack RAG retriever → BEIR metrics

We:
1. Load FIQA into a Haystack `DocumentStore`.
2. Use BM25Retriever.
3. Convert results into BEIR-style `results[qid] = {doc_id: score}`.
4. Use BEIR's `EvaluateRetrieval` just for metrics.


In [6]:

try:
    # Haystack v2 API
    from haystack import Document
    from haystack.document_stores import InMemoryDocumentStore
    from haystack.components.retrievers import BM25Retriever

    def build_haystack_store_from_corpus(corpus):
        """
        corpus: BEIR corpus dict {doc_id: {"title": ..., "text": ...}}
        """
        docs = []
        for doc_id, doc in corpus.items():
            text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
            # Haystack v2 uses Document objects
            docs.append(Document(
                content=text,
                meta={"doc_id": doc_id}
            ))
        store = InMemoryDocumentStore()
        store.write_documents(docs)
        return store

    def haystack_results_to_beir_format(corpus, queries, retriever, top_k=10):
        """
        Returns BEIR-style results: results[qid] = {doc_id: score}
        """
        results = {}
        for qid, qtext in queries.items():
            # Haystack v2: retriever.run() returns a dict with 'documents' key
            output = retriever.run(query=qtext, top_k=top_k)
            hits = output.get("documents", [])
            scored_docs = {}
            for rank, d in enumerate(hits):
                doc_id = d.meta.get("doc_id", "")
                score = getattr(d, "score", None)
                if score is None:
                    score = 1.0 / (rank + 1)
                scored_docs[doc_id] = float(score)
            results[qid] = scored_docs
        return results

    def eval_haystack_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        metrics = []
        store = build_haystack_store_from_corpus(corpus)
        # Haystack v2: BM25Retriever takes store as parameter
        haystack_bm25 = BM25Retriever(document_store=store)

        hs_results = haystack_results_to_beir_format(
            corpus, queries, haystack_bm25, top_k=max(k_values)
        )

        dummy_eval = EvaluateRetrieval(None)
        hs_ndcg, hs_map, hs_recall, hs_prec = dummy_eval.evaluate(
            qrels, hs_results, k_values
        )
        metrics.append(
            extract_metrics_dict("haystack_bm25", hs_ndcg, hs_map, hs_recall, hs_prec, k_values)
        )
        return metrics

except ImportError as e:
    print(f"Haystack not installed or import error: {e}. Skipping Haystack pipeline.")
    def eval_haystack_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        return []


Haystack not installed or import error: cannot import name 'InMemoryDocumentStore' from 'haystack.document_stores' (/home/kzy816/.conda/envs/rag_app/lib/python3.11/site-packages/haystack/document_stores/__init__.py). Skipping Haystack pipeline.



## 4. LangChain FAISS retriever → BEIR metrics

We:
1. Build a FAISS VectorStore from FIQA.
2. Use `as_retriever()` to get relevant documents.
3. Convert outputs to BEIR-style results.


In [7]:

try:
    from langchain_community.vectorstores import FAISS
    from langchain_community.embeddings import HuggingFaceEmbeddings

    def build_langchain_faiss_from_corpus(corpus):
        texts = []
        metadatas = []
        for doc_id, doc in corpus.items():
            text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
            texts.append(text)
            metadatas.append({"doc_id": doc_id})
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/msmarco-distilbert-base-tas-b"
        )
        vs = FAISS.from_texts(texts, embedding=embeddings, metadatas=metadatas)
        return vs

    def langchain_results_to_beir_format(corpus, queries, retriever, top_k=10):
        results = {}
        for qid, qtext in queries.items():
            # LangChain v0.2+ uses invoke() instead of get_relevant_documents()
            try:
                docs = retriever.invoke(qtext)
            except (AttributeError, TypeError):
                docs = retriever.get_relevant_documents(qtext)
            docs = docs[:top_k]
            scored_docs = {}
            for rank, d in enumerate(docs):
                doc_id = d.metadata["doc_id"]
                score = d.metadata.get("score", 1.0/(rank+1))
                scored_docs[doc_id] = float(score)
            results[qid] = scored_docs
        return results

    def eval_langchain_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        metrics = []
        vs = build_langchain_faiss_from_corpus(corpus)
        retriever = vs.as_retriever(search_kwargs={"k": max(k_values)})

        lc_results = langchain_results_to_beir_format(
            corpus, queries, retriever, top_k=max(k_values)
        )
        dummy_eval = EvaluateRetrieval(None)
        lc_ndcg, lc_map, lc_recall, lc_prec = dummy_eval.evaluate(
            qrels, lc_results, k_values
        )
        metrics.append(
            extract_metrics_dict("langchain_faiss", lc_ndcg, lc_map, lc_recall, lc_prec, k_values)
        )
        return metrics

except ImportError:
    def eval_langchain_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        print("LangChain / FAISS not installed, skipping LangChain pipeline.")
        return []



## 5. LlamaIndex VectorStoreIndex retriever → BEIR metrics

We:
1. Build a `VectorStoreIndex` from FIQA.
2. Use `as_retriever()` to retrieve nodes.
3. Convert to BEIR-style results.


In [8]:

try:
    from llama_index.core import Document as LIDocument
    from llama_index.core import VectorStoreIndex
    
    # Use LlamaIndex's native HuggingFace embeddings
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding

    def build_llamaindex_index_from_corpus(corpus):
        print(f"📚 Building LlamaIndex from {len(corpus)} documents...")
        docs = []
        for doc_id, doc in corpus.items():
            text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
            docs.append(
                LIDocument(
                    text=text,
                    metadata={"doc_id": doc_id}
                )
            )
        print(f"✅ Created {len(docs)} LlamaIndex documents")
        
        # Use HuggingFace embedding model (same model as other pipelines)
        print("🔄 Loading embedding model...")
        embed_model = HuggingFaceEmbedding(
            model_name="sentence-transformers/msmarco-distilbert-base-tas-b"
        )
        print("✅ Embedding model loaded")
        
        # Build index with progress indicator
        # This will compute embeddings for all documents - can take a while!
        print("🔄 Computing embeddings and building index (this may take several minutes)...")
        print(f"   Processing {len(docs)} documents...")
        index = VectorStoreIndex.from_documents(
            docs, 
            embed_model=embed_model,
            show_progress=True  # Show progress bar
        )
        print("✅ Index built successfully")
        return index

    def llamaindex_results_to_beir_format(corpus, queries, retriever, top_k=10):
        results = {}
        for qid, qtext in queries.items():
            nodes = retriever.retrieve(qtext)
            nodes = nodes[:top_k]
            scored_docs = {}
            for rank, n in enumerate(nodes):
                doc_id = n.metadata["doc_id"]
                score = getattr(n, "score", None)
                if score is None:
                    score = getattr(n, "similarity", 1.0/(rank+1))
                scored_docs[doc_id] = float(score)
            results[qid] = scored_docs
        return results

    def eval_llamaindex_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        metrics = []
        index = build_llamaindex_index_from_corpus(corpus)
        retriever = index.as_retriever(similarity_top_k=max(k_values))

        li_results = llamaindex_results_to_beir_format(
            corpus, queries, retriever, top_k=max(k_values)
        )
        dummy_eval = EvaluateRetrieval(None)
        li_ndcg, li_map, li_recall, li_prec = dummy_eval.evaluate(
            qrels, li_results, k_values
        )
        metrics.append(
            extract_metrics_dict("llamaindex_vector", li_ndcg, li_map, li_recall, li_prec, k_values)
        )
        return metrics

except ImportError:
    def eval_llamaindex_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        print("LlamaIndex not installed, skipping LlamaIndex pipeline.")
        return []



## 6. Run all pipelines and compare


## Pipeline Comparison: Why Metrics Differ

### 1. **beir_bm25_rankbm25** (Lexical/Keyword Search)
**How it works:**
- Uses **BM25 algorithm** - a probabilistic ranking function for text search
- **Token-based matching**: Breaks documents and queries into tokens (words)
- Scores documents based on **term frequency** and **inverse document frequency** (TF-IDF-like)
- Matches exact keywords and their variations

**Strengths:**
- ✅ Fast and efficient (no neural networks)
- ✅ Excellent for exact keyword matches
- ✅ Works well when query terms appear in relevant documents
- ✅ No GPU needed, pure Python implementation

**Weaknesses:**
- ❌ Cannot handle **semantic similarity** (e.g., "car" vs "automobile")
- ❌ Misses documents with **synonyms** or **paraphrases**
- ❌ Struggles with **conceptual queries** that don't use exact keywords

**Why metrics are lower:**
- FIQA (Financial Q&A) often has queries that need semantic understanding
- Example: Query "investment risk" might miss documents using "portfolio volatility"
- BM25 only finds documents with exact word matches

---

### 2. **beir_dense_sbert** (Semantic Search)
**How it works:**
- Uses **Sentence-BERT embeddings** (`msmarco-distilbert-base-tas-b`)
- Converts documents and queries into **dense vector embeddings** (768 dimensions)
- Uses **cosine similarity** to find semantically similar documents
- Trained on MS MARCO dataset (information retrieval)

**Strengths:**
- ✅ Understands **semantic meaning**, not just keywords
- ✅ Finds documents with **synonyms** and **paraphrases**
- ✅ Handles **conceptual queries** well
- ✅ Better for domain-specific language (financial terminology)

**Weaknesses:**
- ❌ Slower than BM25 (requires neural network inference)
- ❌ May miss exact keyword matches in some cases
- ❌ Requires GPU for fast inference (though CPU works)

**Why metrics are highest:**
- FIQA queries benefit from semantic understanding
- Can match "investment risk" with "portfolio volatility" or "market exposure"
- The model was trained specifically for information retrieval tasks

---

### 3. **langchain_faiss** (Semantic Search with FAISS)
**How it works:**
- Uses the **same embedding model** as `beir_dense_sbert` (`msmarco-distilbert-base-tas-b`)
- Stores embeddings in **FAISS** (Facebook AI Similarity Search) vector database
- Uses **FAISS index** for fast approximate nearest neighbor search
- Same semantic search approach, different implementation

**Strengths:**
- ✅ Same semantic understanding as `beir_dense_sbert`
- ✅ **FAISS optimization** for faster retrieval on large datasets
- ✅ Efficient vector storage and indexing
- ✅ Part of LangChain ecosystem (easy integration)

**Weaknesses:**
- ❌ Slightly lower metrics than BEIR's dense search (implementation differences)
- ❌ FAISS uses approximate search (trade-off between speed and accuracy)
- ❌ Different scoring/normalization may affect results

**Why metrics are between BM25 and Dense SBERT:**
- Uses same embedding model, so should perform similarly to `beir_dense_sbert`
- Small differences likely due to:
  - **FAISS approximate search** vs exact search
  - **Different scoring mechanisms** (LangChain vs BEIR)
  - **Index building differences** (how vectors are stored/retrieved)

---

### Key Insights from Your Results:

Based on your metrics:
- **beir_dense_sbert**: Best performance (nDCG@5 = 0.2124)
- **langchain_faiss**: Close second (nDCG@5 = 0.1922) 
- **beir_bm25_rankbm25**: Lowest (nDCG@5 = 0.1782)

**Why Dense SBERT > LangChain FAISS:**
- BEIR's implementation uses **exact similarity search**
- FAISS uses **approximate nearest neighbor** (faster but slightly less accurate)
- Different normalization/scoring in retrieval process

**Why Both Semantic > BM25:**
- FIQA queries require understanding financial concepts, not just keywords
- Semantic search captures relationships like "risk" ↔ "volatility" ↔ "uncertainty"
- BM25 misses these semantic connections

**Recommendation:**
- For **production RAG systems**: Use **hybrid search** (BM25 + Dense) for best of both worlds
- For **speed-critical**: Use BM25 or FAISS
- For **accuracy-critical**: Use exact dense search (like BEIR's implementation)


In [9]:

# This may take a while the first time (downloads FIQA, builds indexes, computes embeddings).

corpus, queries, qrels = load_fiqa(split="test")

all_metrics = []

# 1) BEIR-native
print("Running BEIR-native pipelines...")
all_metrics.extend(eval_beir_pipelines(corpus, queries, qrels))

# 2) Haystack
print("Running Haystack pipeline...")
all_metrics.extend(eval_haystack_pipeline(corpus, queries, qrels))

# 3) LangChain
print("Running LangChain pipeline...")
all_metrics.extend(eval_langchain_pipeline(corpus, queries, qrels))

# 4) LlamaIndex
print("Running LlamaIndex pipeline...")
all_metrics.extend(eval_llamaindex_pipeline(corpus, queries, qrels))

print("\n=== Comparison Table ===")
print_metrics_table(all_metrics, k_values=K_VALUES)


datasets/fiqa.zip: 100%|██████████| 17.1M/17.1M [00:03<00:00, 5.26MiB/s]
100%|██████████| 57638/57638 [00:00<00:00, 333445.00it/s]


Running BEIR-native pipelines...
⚠️ Elasticsearch BM25 failed: 'tuple' object has no attribute 'split'. Using rank-bm25 fallback.
Found P@5 with value 0.08148
Found Recall@5 with value 0.19135
Found NDCG@5 with value 0.17815
Found MAP@5 with value 0.13678
Found P@10 with value 0.05648
Found Recall@10 with value 0.25843
Found NDCG@10 with value 0.20164
Found MAP@10 with value 0.14914
✅ BM25 using rank-bm25 (pure Python)


Batches: 100%|██████████| 60/60 [00:01<00:00, 47.97it/s]


Found P@5 with value 0.09784
Found Recall@5 with value 0.22595
Found NDCG@5 with value 0.21238
Found MAP@5 with value 0.16576
Found P@10 with value 0.06389
Found Recall@10 with value 0.29024
Found NDCG@10 with value 0.23413
Found MAP@10 with value 0.1777
✅ Dense SBERT pipeline completed
Found P@5 with value 0.10988
Found Recall@5 with value 0.25894
Found NDCG@5 with value 0.23132
Found MAP@5 with value 0.17738
Found P@10 with value 0.0713
Found Recall@10 with value 0.32537
Found NDCG@10 with value 0.25373
Found MAP@10 with value 0.19015
✅ Hybrid pipeline completed
Running Haystack pipeline...
Running LangChain pipeline...


/tmp/ipykernel_2593891/2453054080.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Found P@5 with value 0.08951
Found Recall@5 with value 0.20572
Found NDCG@5 with value 0.19224
Found MAP@5 with value 0.14761
Found P@10 with value 0.05725
Found Recall@10 with value 0.26006
Found NDCG@10 with value 0.20988
Found MAP@10 with value 0.1577
Running LlamaIndex pipeline...
📚 Building LlamaIndex from 57638 documents...
✅ Created 57638 LlamaIndex documents
🔄 Loading embedding model...
✅ Embedding model loaded
🔄 Computing embeddings and building index (this may take several minutes)...
   Processing 57638 documents...


Generating embeddings: 100%|██████████| 618/618 [00:00<00:00, 620.01it/s]


✅ Index built successfully
Found P@5 with value 0.10864
Found Recall@5 with value 0.25835
Found NDCG@5 with value 0.23352
Found MAP@5 with value 0.18079
Found P@10 with value 0.07068
Found Recall@10 with value 0.31905
Found NDCG@10 with value 0.25458
Found MAP@10 with value 0.19344

=== Comparison Table ===
pipeline            P@5     R@5     nDCG@5  P@10    R@10    nDCG@10  
---------------------------------------------------------------------
beir_bm25_rankbm25  0.0815  0.1913  0.1782  0.0565  0.2584  0.2016   
beir_dense_sbert    0.0978  0.2260  0.2124  0.0639  0.2902  0.2341   
beir_hybrid_0.5     0.1099  0.2589  0.2313  0.0713  0.3254  0.2537   
langchain_faiss     0.0895  0.2057  0.1922  0.0573  0.2601  0.2099   
llamaindex_vector   0.1086  0.2584  0.2335  0.0707  0.3191  0.2546   


In [32]:

# This may take a while the first time (downloads FIQA, builds indexes, computes embeddings).

corpus, queries, qrels = load_fiqa(split="test")

all_metrics = []

# 1) BEIR-native
print("Running BEIR-native pipelines...")
all_metrics.extend(eval_beir_pipelines(corpus, queries, qrels))

# 2) Haystack
print("Running Haystack pipeline...")
all_metrics.extend(eval_haystack_pipeline(corpus, queries, qrels))

# 3) LangChain
print("Running LangChain pipeline...")
all_metrics.extend(eval_langchain_pipeline(corpus, queries, qrels))

# 4) LlamaIndex
print("Running LlamaIndex pipeline...")
all_metrics.extend(eval_llamaindex_pipeline(corpus, queries, qrels))

print("\n=== Comparison Table ===")
print_metrics_table(all_metrics, k_values=K_VALUES)


100%|██████████| 57638/57638 [00:00<00:00, 347862.61it/s]


Running BEIR-native pipelines...
⚠️ Elasticsearch BM25 failed: ElasticSearch.__init__() got an unexpected keyword argument 'index_name'
Found P@5 with value 0.08148
Found Recall@5 with value 0.19135
Found NDCG@5 with value 0.17815
Found MAP@5 with value 0.13678
Found P@10 with value 0.05648
Found Recall@10 with value 0.25843
Found NDCG@10 with value 0.20164
Found MAP@10 with value 0.14914
✅ BM25 using rank-bm25 (pure Python)


Batches: 100%|██████████| 60/60 [00:01<00:00, 47.25it/s]


Found P@5 with value 0.09784
Found Recall@5 with value 0.22595
Found NDCG@5 with value 0.21238
Found MAP@5 with value 0.16576
Found P@10 with value 0.06389
Found Recall@10 with value 0.29024
Found NDCG@10 with value 0.23413
Found MAP@10 with value 0.1777
✅ Dense SBERT pipeline completed
⚠️ Hybrid pipeline failed: 'EvaluateRetrieval' object has no attribute 'hybrid'
Running Haystack pipeline...
Running LangChain pipeline...
Found P@5 with value 0.08951
Found Recall@5 with value 0.20572
Found NDCG@5 with value 0.19224
Found MAP@5 with value 0.14761
Found P@10 with value 0.05725
Found Recall@10 with value 0.26006
Found NDCG@10 with value 0.20988
Found MAP@10 with value 0.1577
Running LlamaIndex pipeline...
LlamaIndex not installed, skipping LlamaIndex pipeline.

=== Comparison Table ===
pipeline	P@5	R@5	nDCG@5	P@10	R@10	nDCG@10
beir_bm25_rankbm25	0.0815	0.1913	0.1782	0.0565	0.2584	0.2016
beir_dense_sbert	0.0978	0.2260	0.2124	0.0639	0.2902	0.2341
langchain_faiss	0.0895	0.2057	0.1922	0.0573

In [42]:
print("\n=== Comparison Table ===")
print_metrics_table(all_metrics, k_values=K_VALUES)


=== Comparison Table ===
pipeline            P@5     R@5     nDCG@5  P@10    R@10    nDCG@10  
---------------------------------------------------------------------
beir_bm25_rankbm25  0.0815  0.1913  0.1782  0.0565  0.2584  0.2016   
beir_dense_sbert    0.0978  0.2260  0.2124  0.0639  0.2902  0.2341   
langchain_faiss     0.0895  0.2057  0.1922  0.0573  0.2601  0.2099   
